# BPI2019 — 01 · Clustering feature re-engineering (EDA)

**Goal:** find an **interpretable** behavioural feature set (no dimensionality reduction) that yields
**silhouette ≥ 0.4** (aim 0.4–0.5) with **5–8 clusters**, and whose cohorts are genuinely
behaviourally meaningful (not a trivial 1–2-feature split).

Approach: engineer a richer/cleaner behavioural feature set (structural flags + counts + ratios +
selected bigrams), inspect distributions/variance/correlation, then run a **bounded** search over
{feature subset × scaling} × k = 5..8 and keep the best. Reuses the cached BPI2019 tables.

## 1. Load + engineer behavioural features

In [1]:
import os, json, warnings, time

warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler, RobustScaler
from sklearn.metrics import silhouette_score

SEED, SIL_SAMPLE = 42, 20000
KEY, ACT = "case:concept:name", "concept:name"
REPO = os.path.abspath(os.path.join(os.getcwd(), ".."))  # repository root (parent of notebooks/)
INTERIM = os.path.join(REPO, "data", "interim")
OUT = os.path.join(REPO, "artifacts")
os.makedirs(OUT, exist_ok=True)

proc = pd.read_csv(
    os.path.join(INTERIM, "bpi2019_process_only.csv")
)  # 11 engineered + 30 bigrams
ENG11 = [
    "total_events",
    "unique_activities",
    "n_resources",
    "n_goods_receipts",
    "has_payment_block_removed",
    "has_change_activity",
    "has_cancel_activity",
    "has_delete_po_item",
    "automation_ratio",
    "repetition_ratio",
    "handoff_ratio",
]
BG = [c for c in proc.columns if c.startswith("bg_")]

# --- engineer extra STRUCTURAL behavioural flags/counts from the event log ---
ev = pd.read_parquet(
    os.path.join(INTERIM, "bpi2019_events.parquet"), columns=[KEY, ACT]
)
a = ev[ACT]
ev2 = pd.DataFrame(
    {
        KEY: ev[KEY],
        "is_gr": (a == "Record Goods Receipt").values,
        "is_inv": (a == "Record Invoice Receipt").values,
        "is_clear": (a == "Clear Invoice").values,
        "is_srm": a.str.startswith("SRM:").values,
        "is_service": (a == "Record Service Entry Sheet").values,
        "is_ordconf": (a == "Receive Order Confirmation").values,
        "is_vinv": (a == "Vendor creates invoice").values,
    }
)
gs = ev2.groupby(KEY).sum()
struct = pd.DataFrame(index=gs.index)
struct["n_invoice_receipts"] = gs["is_inv"]
struct["n_clear"] = gs["is_clear"]
struct["has_srm"] = (gs["is_srm"] > 0).astype(int)
struct["has_service_entry"] = (gs["is_service"] > 0).astype(int)
struct["has_order_confirmation"] = (gs["is_ordconf"] > 0).astype(int)
struct["has_goods_receipt"] = (gs["is_gr"] > 0).astype(int)
struct["has_vendor_invoice"] = (gs["is_vinv"] > 0).astype(int)
struct = struct.reset_index()

B = proc.merge(struct, on=KEY, how="left").fillna(0)
STRUCT_EXTRA = [
    "n_invoice_receipts",
    "n_clear",
    "has_srm",
    "has_service_entry",
    "has_order_confirmation",
    "has_goods_receipt",
    "has_vendor_invoice",
]
print(
    "B",
    B.shape,
    "| engineered",
    len(ENG11),
    "| bigrams",
    len(BG),
    "| extra struct",
    len(STRUCT_EXTRA),
)

B (251734, 49) | engineered 11 | bigrams 30 | extra struct 7


## 2. EDA — variance, skew, correlation (decide log-scaling)

In [2]:
NUM = [
    "total_events",
    "unique_activities",
    "n_resources",
    "n_goods_receipts",
    "n_invoice_receipts",
    "n_clear",
    "automation_ratio",
    "repetition_ratio",
    "handoff_ratio",
]
FLAGS = [
    "has_payment_block_removed",
    "has_change_activity",
    "has_cancel_activity",
    "has_delete_po_item",
    "has_srm",
    "has_service_entry",
    "has_order_confirmation",
    "has_goods_receipt",
    "has_vendor_invoice",
]
desc = B[NUM].describe(percentiles=[0.5, 0.95, 0.99]).T
desc["skew"] = B[NUM].skew()
print("Numeric behavioural features (skew guides log-scaling):")
print(desc[["mean", "50%", "95%", "max", "skew"]].round(2).to_string())
print("\nFlag prevalence:")
print(B[FLAGS].mean().round(3).to_string())
# heavy right-skew counts -> log1p candidates
LOGCOLS = [c for c in NUM if B[c].skew() > 2]
print("\nlog1p candidates (skew>2):", LOGCOLS)

Numeric behavioural features (skew guides log-scaling):
                    mean  50%   95%     max   skew
total_events        6.34  5.0  9.00  990.00  22.13
unique_activities   5.19  5.0  7.00   20.00   0.03
n_resources         4.91  5.0  7.00   22.00  -0.73
n_goods_receipts    1.25  1.0  2.00  269.00  24.56
n_invoice_receipts  0.91  1.0  1.00  279.00  99.06
n_clear             0.77  1.0  1.00   93.00  29.05
automation_ratio    0.24  0.2  0.50    1.00   1.43
repetition_ratio    0.03  0.0  0.25    0.99   4.93
handoff_ratio       0.92  1.0  1.00    1.00  -2.60

Flag prevalence:
has_payment_block_removed    0.222
has_change_activity          0.126
has_cancel_activity          0.033
has_delete_po_item           0.035
has_srm                      0.006
has_service_entry            0.022
has_order_confirmation       0.127
has_goods_receipt            0.931
has_vendor_invoice           0.834

log1p candidates (skew>2): ['total_events', 'n_goods_receipts', 'n_invoice_receipts', 'n_clear', 're

## 3. Bounded clustering search — {feature set × scaling} × k = 5..8

In [3]:
BG10 = (
    B[BG].var().sort_values(ascending=False).head(10).index.tolist()
)  # top-10 bigrams by variance
STRUCT = NUM + FLAGS  # interpretable structural set (16 feats)


def build(cols, logcols=(), scaler="z"):
    X = B[cols].astype(float).copy()
    for c in logcols:
        if c in X.columns:
            X[c] = np.log1p(X[c].clip(lower=0))
    S = RobustScaler() if scaler == "robust" else StandardScaler()
    return S.fit_transform(X)


SPECS = {
    "A_current(eng11+bg30)": (ENG11 + BG, [], "z"),
    "B_eng11_only": (ENG11, [], "z"),
    "C_struct16": (STRUCT, [], "z"),
    "C_struct16_log": (STRUCT, LOGCOLS, "z"),
    "D_struct16_log+bg10": (STRUCT + BG10, LOGCOLS, "z"),
    "E_flags_only": (FLAGS, [], "z"),
}


def sil(X, lab):
    return float(
        silhouette_score(X, lab, sample_size=min(SIL_SAMPLE, len(X)), random_state=SEED)
    )


rows = []
t = time.time()
for name, (cols, logc, sc) in SPECS.items():
    X = build(cols, logc, sc)
    for k in range(5, 9):
        lab = KMeans(n_clusters=k, n_init=5, random_state=SEED).fit_predict(X)
        rows.append(
            {
                "spec": name,
                "n_feats": X.shape[1],
                "k": k,
                "silhouette": round(sil(X, lab), 3),
            }
        )
    print(f"  done {name} [{time.time()-t:.0f}s]", flush=True)
res = pd.DataFrame(rows)
piv = res.pivot(index="spec", columns="k", values="silhouette")
print("\nSilhouette by spec × k:")
print(piv.to_string())
best = res.loc[res.silhouette.idxmax()]
print(f"\nBEST: {best.spec} @ k={int(best.k)} → silhouette {best.silhouette}")
res.to_csv(os.path.join(OUT, "clustering_feature_search.csv"), index=False)

  done A_current(eng11+bg30) [21s]
  done B_eng11_only [35s]
  done C_struct16 [49s]
  done C_struct16_log [63s]
  done D_struct16_log+bg10 [77s]
  done E_flags_only [90s]

Silhouette by spec × k:
k                          5      6      7      8
spec                                             
A_current(eng11+bg30)  0.280  0.301  0.331  0.350
B_eng11_only           0.427  0.478  0.537  0.547
C_struct16             0.387  0.491  0.404  0.411
C_struct16_log         0.386  0.380  0.393  0.450
D_struct16_log+bg10    0.370  0.389  0.406  0.415
E_flags_only           0.523  0.564  0.659  0.709

BEST: E_flags_only @ k=8 → silhouette 0.709


## 4. Profile the best config (confirm cohorts are behaviourally meaningful)

In [4]:
# Selection rationale: dropping the 30 sparse bigrams is the key lever (A 0.28-0.35 -> B 0.43-0.55).
# E_flags_only scores highest (0.52-0.71) but clusters only on event presence/absence (shallow); we
# prefer B_eng11_only (counts + ratios + rework flags) - richer, interpretable, clears the 0.4-0.5 target.
FINAL_SPEC = "B_eng11_only"
cols, logc, sc = SPECS[FINAL_SPEC]
Xb = build(cols, logc, sc)
for k in (6, 7):
    lab = KMeans(n_clusters=k, n_init=10, random_state=SEED).fit_predict(Xb)
    tmp = B.copy()
    tmp["c"] = lab
    prof = tmp.groupby("c")[cols].mean().round(2)
    prof.insert(
        0,
        "pct",
        (100 * tmp.c.value_counts(normalize=True).sort_index()).round(1).values,
    )
    print(f"\n=== {FINAL_SPEC} @ k={k} | silhouette {sil(Xb, lab):.3f} ===")
    print(prof.to_string())

FINAL_K = (
    6  # good balance: silhouette ~0.48 (>=0.4), 6 clusters (kinder to classification)
)
lab = KMeans(n_clusters=FINAL_K, n_init=10, random_state=SEED).fit_predict(Xb)
B["cohort"] = lab
B[[KEY, "cohort"]].to_csv(os.path.join(OUT, "cohort_labels.csv"), index=False)
json.dump(
    {
        "spec": FINAL_SPEC,
        "k": FINAL_K,
        "silhouette": round(sil(Xb, lab), 3),
        "columns": cols,
        "logcols": list(logc),
        "scaler": sc,
        "note": "dropped 30 bigram TF-IDF (they depressed silhouette); engineered-only, z-scored",
    },
    open(os.path.join(OUT, "clustering_best_config.json"), "w"),
    indent=2,
)
print(
    f"\nSAVED clustering_best_config.json + cohort_labels.csv | FINAL: {FINAL_SPEC} @ k={FINAL_K}"
)


=== B_eng11_only @ k=6 | silhouette 0.478 ===
    pct  total_events  unique_activities  n_resources  n_goods_receipts  has_payment_block_removed  has_change_activity  has_cancel_activity  has_delete_po_item  automation_ratio  repetition_ratio  handoff_ratio
c                                                                                                                                                                                                                  
0  58.7          5.36               5.31         5.14              1.02                       0.00                 0.10                 0.00                 0.0              0.26              0.01           0.97
1   3.5          2.68               2.63         1.69              0.04                       0.00                 0.11                 0.01                 1.0              0.08              0.01           0.63
2   5.8         14.34               6.95         5.56              2.86                       0.31       

## 5. Behavioural explanation — top distinctive paths per cohort

The bigram TF-IDF features are **not** thrown away: they degrade the *clustering distance* (sparse,
high-dimensional → curse of dimensionality), but they are exactly the right tool to **explain** the
cohorts *after* clustering. This is a clean decoupling — cluster on the compact behavioural summary
(counts / ratios / rework flags), then narrate each cohort with the process paths it over-uses.

Two complementary views per cohort:
1. **Distinctive activity transitions** — aggregate the 30 bigram TF-IDF, rank by *lift* (cohort mean ÷
   global mean). Answers "which control-flow steps make this cohort different?".
2. **Top concrete trace variants** — the most frequent full activity sequences in the cohort, straight
   from the event log. Answers "what do these cases actually look like end-to-end?".


In [6]:
# View 1: distinctive activity transitions per cohort (bigram TF-IDF aggregated, ranked by lift)
gmean = B[BG].mean()  # global mean TF-IDF per transition
rows_tr = []
for c in sorted(B["cohort"].unique()):
    n = int((B.cohort == c).sum())
    m = B.loc[B.cohort == c, BG].mean()
    d = pd.DataFrame({"cohort_mean": m, "global_mean": gmean})
    d["lift"] = (d.cohort_mean + 1e-9) / (d.global_mean + 1e-9)
    # keep transitions that are actually present in the cohort (avoid noisy lift on ~0 means)
    d = d[d.cohort_mean > 0.02].sort_values("lift", ascending=False).head(5)
    print(
        f"\n=== Cohort {c}  (n={n:,}, {100*n/len(B):.1f}%) — distinctive transitions ==="
    )
    for name, r in d.iterrows():
        print(
            f"   {r.lift:5.1f}x  {name[3:]:58s}  (mean {r.cohort_mean:.3f} vs {r.global_mean:.3f})"
        )
        rows_tr.append(
            {
                "cohort": c,
                "transition": name[3:],
                "cohort_mean": round(r.cohort_mean, 4),
                "global_mean": round(r.global_mean, 4),
                "lift": round(r.lift, 2),
            }
        )
pd.DataFrame(rows_tr).to_csv(
    os.path.join(OUT, "cohort_path_transitions.csv"), index=False
)
print("\nsaved cohort_path_transitions.csv")


=== Cohort 0  (n=147,762, 58.7%) — distinctive transitions ===
     1.6x  Record Invoice Receipt -> Clear Invoice                     (mean 0.329 vs 0.201)
     1.4x  Vendor creates invoice -> Record Goods Receipt              (mean 0.254 vs 0.184)
     1.4x  Record Goods Receipt -> Vendor creates invoice              (mean 0.193 vs 0.141)
     1.4x  Record Goods Receipt -> Record Invoice Receipt              (mean 0.255 vs 0.186)
     1.3x  Receive Order Confirmation -> Record Goods Receipt          (mean 0.063 vs 0.047)

=== Cohort 1  (n=8,751, 3.5%) — distinctive transitions ===
    28.7x  Create Purchase Order Item -> Delete Purchase Order Item    (mean 0.763 vs 0.027)
     1.9x  Create Purchase Order Item -> Receive Order Confirmation    (mean 0.119 vs 0.064)
     1.0x  Create Purchase Requisition Item -> Create Purchase Order Item  (mean 0.102 vs 0.100)
     0.9x  Create Purchase Order Item -> Change Quantity               (mean 0.023 vs 0.027)

=== Cohort 2  (n=14,492, 5.8%) — 

In [7]:
# View 2: top concrete trace variants per cohort (full activity sequence from the event log)
seq = ev.groupby(KEY, sort=False)[ACT].agg(" -> ".join).rename("variant").reset_index()
vdf = seq.merge(B[[KEY, "cohort"]], on=KEY, how="inner")
rows_var = []
for c in sorted(vdf.cohort.unique()):
    sub = vdf[vdf.cohort == c]
    top = sub.variant.value_counts().head(5)
    print(
        f"\n=== Cohort {c}  ({len(sub):,} cases, {sub.variant.nunique():,} distinct variants) — top paths ==="
    )
    for v, cnt in top.items():
        disp = v if len(v) <= 130 else v[:127] + "..."
        print(f"   {100*cnt/len(sub):5.1f}%  {disp}")
        rows_var.append(
            {
                "cohort": c,
                "pct": round(100 * cnt / len(sub), 1),
                "n_events": v.count("->") + 1,
                "variant": v,
            }
        )
pd.DataFrame(rows_var).to_csv(os.path.join(OUT, "cohort_top_variants.csv"), index=False)
print("\nsaved cohort_top_variants.csv")


=== Cohort 0  (147,762 cases, 1,673 distinct variants) — top paths ===
    34.0%  Create Purchase Order Item -> Vendor creates invoice -> Record Goods Receipt -> Record Invoice Receipt -> Clear Invoice
    20.8%  Create Purchase Order Item -> Record Goods Receipt -> Vendor creates invoice -> Record Invoice Receipt -> Clear Invoice
     6.6%  Create Purchase Order Item -> Receive Order Confirmation -> Record Goods Receipt -> Vendor creates invoice -> Record Invoice Re...
     6.0%  Create Purchase Requisition Item -> Create Purchase Order Item -> Vendor creates invoice -> Record Goods Receipt -> Record Invo...
     2.9%  Create Purchase Order Item -> Receive Order Confirmation -> Vendor creates invoice -> Record Goods Receipt -> Record Invoice Re...

=== Cohort 1  (8,751 cases, 220 distinct variants) — top paths ===
    60.5%  Create Purchase Order Item -> Delete Purchase Order Item
    11.4%  Create Purchase Requisition Item -> Create Purchase Order Item -> Delete Purchase Order Item
